# Exploratory Data Analysis on the Marathos Dataset Part 2
- Year of event
- Event dates
- Event name
- Event number of finishers


## Loading the dataset

In [0]:
# Use the raw dataset inside the volume
VOLUME_PATH = "/Volumes/marathos_catalog/default/raw/"

# Use spark and sql
spark.sql(f"LIST '{VOLUME_PATH}'")

In [0]:
# See data path
DATA_PATH = "/Volumes/marathos_catalog/default/raw/data/"

display(spark.sql(f"LIST '{DATA_PATH}'"))

In [0]:
# Show what is inside the data folder. Using this method for better debugging later.
FILE_PATH = "/Volumes/marathos_catalog/default/raw/data/TWO_CENTURIES_OF_UM_RACES.csv"

# read csv
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(FILE_PATH)
)

display(df)

## Schema Check

In [0]:
# Check the schema that Spark inferred from the CSV file | column name + data type
from pyspark.sql.functions import col, count, when
df.printSchema()

## Year of event
### Check for NULLS

In [0]:
# Check null values in Year of event

year_of_event_null_count = (
    df
    .filter(col("Year of event").isNull())
    .count()
)

print(f"Null values in Year of event: {year_of_event_null_count}")

### Year of event descriptive analysis

In [0]:
# Descriptive analysis of Year of event

display(
    df
    .select("Year of event")
    .summary()
)

In [0]:
# Number of records per event year
year_distribution_df = (
    df
    .groupBy("Year of event")
    .agg(count("*").alias("row_count"))
    .orderBy(col("Year of event").asc())
)

display(year_distribution_df)

## Event dates
### Check NULLS

In [0]:

# Check null values in Event dates

event_dates_null_count = (
    df
    .filter(col("Event dates").isNull())
    .count()
)

print(f"Null values in Event dates: {event_dates_null_count}")

## Distinct Event dates

- The `Event dates` column has **0 null values** and **14,425 distinct values**.
-  The column is currently stored as a string. Most values appear to use a European date format such as `dd.MM.yyyy`, for example `24.06.2017`.
-  During inspection, I also found date ranges such as `12.-13.10.2019` and `07.-08.10.2017`. This means some events happen over more than one day.
-  In the Silver layer, this column should be cleaned into proper date columns, for example:
    - `event_date_raw`
    - `event_start_date`
    - `event_end_date`

- The final cleaned date columns should use Spark's `date` type, with a standard format such as `yyyy-MM-dd`.

In [0]:
# Count distinct Event dates values

distinct_event_dates_count = (
    df
    .select("Event dates")
    .distinct()
    .count()
)

print(f"Number of distinct Event dates values: {distinct_event_dates_count}")

In [0]:
# Show most common Event dates values

event_dates_counts_df = (
    df
    .groupBy("Event dates")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

display(event_dates_counts_df)

In [0]:
# Classify Event dates formats

event_dates_format_df = (
    df
    .withColumn(
        "event_date_format_type",
        when(col("Event dates").rlike(r"^\d{2}\.\d{2}\.\d{4}$"), "single_date_dd_mm_yyyy")
        .when(col("Event dates").rlike(r"^\d{2}\.-\d{2}\.\d{2}\.\d{4}$"), "date_range_same_month")
        .when(col("Event dates").rlike(r"^\d{2}\.\d{2}\.-\d{2}\.\d{2}\.\d{4}$"), "date_range_different_month")
        .otherwise("unknown_format")
    )
    .groupBy("event_date_format_type")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

display(event_dates_format_df)

In [0]:
# Inspect Event dates with unknown format

unknown_event_dates_df = (
    df
    .withColumn(
        "event_date_format_type",
        when(col("Event dates").rlike(r"^\d{2}\.\d{2}\.\d{4}$"), "single_date_dd_mm_yyyy")
        .when(col("Event dates").rlike(r"^\d{2}\.-\d{2}\.\d{2}\.\d{4}$"), "date_range_same_month")
        .when(col("Event dates").rlike(r"^\d{2}\.\d{2}\.-\d{2}\.\d{2}\.\d{4}$"), "date_range_different_month")
        .otherwise("unknown_format")
    )
    .filter(col("event_date_format_type") == "unknown_format")
    .groupBy("Event dates")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

display(unknown_event_dates_df)

### Silver layer plan for `Event dates`
- In the raw dataset, the `Event dates` column is stored as a string. During EDA, I found that the column has **0 null values** and contains several different date formats.

- This shows that some events are single-day events, while others are multi-day events. Because of this, the column should not be converted into only one date column.
    - `24.06.2017`
    - `12.-13.10.2019`
    - `31.12.-01.01.2018`
    - `28.12.2019-03.01.2020`

- In the Silver layer, I should keep the original raw value for traceability and create structured date columns.
    - `event_date_raw`: the original value from the raw dataset
    - `event_date_format_type`: the detected format type
    - `event_start_date`: the parsed start date as a proper Spark `date` type
    - `event_end_date`: the parsed end date as a proper Spark `date` type
    - `event_duration_days`: the number of days between start and end date

- Example transformations:
```
`24.06.2017` → start date: `2017-06-24`, end date: `2017-06-24`
`12.-13.10.2019` → start date: `2019-10-12`, end date: `2019-10-13`
`28.12.2019-03.01.2020` → start date: `2019-12-28`, end date: `2020-01-03`
```

- The cleaned date columns should use Spark's `date` type and follow a standard format such as `yyyy-MM-dd`.
After parsing, I should validate that:
- `event_start_date` is not null
- `event_end_date` is not null
- `event_end_date` is greater than or equal to `event_start_date`

## Event name
### Check NULLS

In [0]:

# Check null values in Event name

event_name_null_count = (
    df
    .filter(col("Event name").isNull())
    .count()
)

print(f"Null values in Event name: {event_name_null_count}")

In [0]:
# Count unique event names

unique_event_name_count = (
    df
    .select("Event name")
    .distinct()
    .count()
)

print(f"Number of unique event names: {unique_event_name_count}")

In [0]:
# Show most common event names

event_name_counts_df = (
    df
    .groupBy("Event name")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

display(event_name_counts_df)

- I check whether the column has missing values, how many unique event names exist, and which events appear most often in the dataset.
- This is important because `Event name` can later be used to create an `event_id` in the Silver layer or dimensional model.

## Event number of finishers
### Check NULLS

In [0]:
# Check null values in Event number of finishers

event_finishers_null_count = (
    df
    .filter(col("Event number of finishers").isNull())
    .count()
)

print(f"Null values in Event number of finishers: {event_finishers_null_count}")

In [0]:
# Descriptive summary of Event number of finishers

display(
    df
    .select("Event number of finishers")
    .summary()
)

In [0]:
# Check rows where Event number of finishers is 0

zero_finishers_count = (
    df
    .filter(col("Event number of finishers") == 0)
    .count()
)

print(f"Rows with 0 finishers: {zero_finishers_count}")

In [0]:
# Inspect rows where Event number of finishers is 0

display(
    df
    .filter(col("Event number of finishers") == 0)
)

### Zero finishers check
- I found **10 rows** where `Event number of finishers` is `0`.
- Since this column represents how many athletes finished an event, a value of `0` may indicate incomplete, invalid, or unusual event records.
- Because there are only 10 such rows in a dataset with over 7 million rows, I will document them during EDA and decide in the Silver layer that they should be removed.